In [1]:
!pip install numpy pandas scikit-learn rapidfuzz sentence-transformers torch

In [2]:
import re
import numpy as np
from rapidfuzz import distance

class HeuristicNoisyIntentClassifier:
    def __init__(self):
        self.categories = [
            "Bill_Inquiries", 
            "Telephone_Number_Request", 
            "Fault_Inquiries", 
            "Technical_Assistant", 
            "Product_And_New_Service", 
            "Add_More_Data"
        ]
        
        # Master Telecom Labeled Patterns Dictionary
        self.domain_patterns = {
            "Bill_Inquiries": {
                "strong": ["බිල", "බිල්පත", "ගෙවීම්", "පේමන්ට්", "bill", "payment", "අමුන්ට්", "amount", "ගෙව්වා", "මුදලක්", "රුපියල්", "ගිණුම්", "එකවුන්ට්", "boc", "රිෆන්ඩ්", "refund"],
                "phrases": ["බිල ගෙව්වා", "කොච්චර ගෙවන්න", "සල්ලි බැන්දා", "බිල් කම්පැනියට"]
            },
            "Telephone_Number_Request": {
                "strong": ["අංකය", "නම්බරය", "දුරකථන", "ලිපිනය", "number", "telephone", "contact", "නම්බරේ", "නම්බර්", "අංකයක්"],
                "phrases": ["නම්බරය දෙනවද", "ජෙනරල් නම්බර්", "සටහන් කරගන්න", "හෙඩ් quarters"]
            },
            "Fault_Inquiries": {
                "strong": ["කැඩිලා", "දෝෂයක්", "ලයිට්", "නිවි නිවි", "රතු", "වැඩකරන්නේනැහැ", "los", "adsl", "router", "රවුටර්", "complain", "කම්ප්ලේන්", "ඉන්වැලිඩ්", "invalid", "චැනල්", "ඇන්ටෙනා"],
                "phrases": ["රතු පාට ලයිට්", "වැඩ කරන්නේ නැහැ", "ලයිට් එක පත්තු", "ඉන්ටර්නෙට් පෙන්නන්නේ නැහැ"]
            },
            "Technical_Assistant": {
                "strong": ["බොක්ස්", "සැකසුම්", "වයර්", "තාක්ෂණික", "සෙටප්", "සෙටින්ග්ස්", "wps", "config", "setup", "cat6", "wired", "cable", "configurations", "extension", "plug", "පවර්"],
                "phrases": ["off කරලා on කරන්න", "router settings", "තාක්ෂණික අංශයට", "WPS එක ඔබලා"]
            },
            "Product_And_New_Service": {
                "strong": ["මාරු", "පැකේජ්", "අලුත්", "වෙනස්", "ලොකේෂන්", "location", "package", "upgrade", "downgrade", "fiber", "ෆයිබර්", "relocate", "unlimited", "ලියුමක්", "ලිපියක්"],
                "phrases": ["ලොකේෂන් මාරු", "පැකේජ් එක change", "අලුත් පැකේජ්", "ස්ථානය මාරු"]
            },
            "Add_More_Data": {
                "strong": ["ඩේටා", "ඉවරයි", "එක්ස්ට්‍රා", "ඇඩ්", "ජීබී", "gb", "data", "extra data", "balance", "බැලන්ස්", "පහක්", "දහයක්"],
                "phrases": ["ජීබී පහක්", "ඩේටා නම් ඉවර", "එක්ස්ට්‍රා ජීබී", "gb කොච්චරක්"]
            }
        }

    def _generate_character_ngrams(self, text, n=3):
        text = re.sub(r'\s+', '', text)
        if len(text) < n:
            return [text]
        return [text[i:i+n] for i in range(len(text) - n + 1)]

    def _calculate_ngram_similarity(self, text_ngrams, target_word):
        target_ngrams = self._generate_character_ngrams(target_word, n=3)
        intersection = set(text_ngrams).intersection(set(target_ngrams))
        if not target_ngrams:
            return 0.0
        return len(intersection) / len(target_ngrams)

    def predict(self, raw_noisy_text):
        normalized_text = str(raw_noisy_text).lower().strip()
        tokens = re.findall(r'\b\w+\b', normalized_text)
        text_ngrams = self._generate_character_ngrams(normalized_text, n=3)
        
        category_scores = {cat: 0.0 for cat in self.categories}
        evidence_logs = {cat: [] for cat in self.categories}
        sim_threshold = 0.76  # Safe standalone variable initialization
        
        for cat in self.categories:
            score = 0.0
            patterns = self.domain_patterns[cat]
            
            # 1. Direct Phrase Weighting Match
            for phrase in patterns["phrases"]:
                if phrase in normalized_text:
                    score += 4.0
                    evidence_logs[cat].append(f"[Phrase] '{phrase}'")
            
            # 2. Strong Core Keyword Hard Anchor Check
            for kw in patterns["strong"]:
                if kw in normalized_text:
                    score += 5.0
                    evidence_logs[cat].append(f"[Exact Keyword] '{kw}'")
                    continue 
                
                # 3. Fuzzy Phone Match Distance Processing Loop
                max_fuzzy_found = 0.0
                best_token = ""
                for token in tokens:
                    sim = distance.JaroWinkler.similarity(kw, token)
                    if sim > sim_threshold:
                        if sim > max_fuzzy_found:
                            max_fuzzy_found = sim
                            best_token = token
                            
                if max_fuzzy_found > 0.0:
                    score += 3.0 * max_fuzzy_found
                    evidence_logs[cat].append(f"[Fuzzy] '{best_token}'->'{kw}' ({max_fuzzy_found:.2f})")
                    
                # 4. Broken Sub-word Combination Extraction Pass
                ngram_sim = self._calculate_ngram_similarity(text_ngrams, kw)
                if ngram_sim > 0.60:
                    score += 2.0 * ngram_sim
                    evidence_logs[cat].append(f"[Ngram Sub-Word] '{kw}' ({ngram_sim:.2f})")
                    
            category_scores[cat] = score

        scores_vector = np.array([category_scores[c] for c in self.categories])
        if np.sum(scores_vector) == 0:
            return {
                "predicted_category": "Bill_Inquiries",
                "confidence_score": 0.1666,
                "matched_evidence": ["Default Safe Flag Triggered"],
                "raw_scoring_matrix": {c: 0.0 for c in self.categories}
            }
            
        exp_scores = np.exp(scores_vector - np.max(scores_vector))
        probabilities = exp_scores / np.sum(exp_scores)
        top_idx = np.argmax(probabilities)
        predicted_category = self.categories[top_idx]
        
        return {
            "predicted_category": predicted_category,
            "confidence_score": round(float(probabilities[top_idx]), 4),
            "matched_evidence": list(set(evidence_logs[predicted_category]))[:4],
            "raw_scoring_matrix": {c: round(category_scores[c], 2) for c in self.categories}
        }

In [3]:
# Initialize the fully offline engine instance
engine = HeuristicNoisyIntentClassifier()

# Your long transcript containing high conversational ambient noise
noisy_input_sample = """රෙස්ට් විසින් හෑන්ඩ් දැනට කනෙක්ෂන් එක බිල්පත නිසා ඩිස්කනෙක්ට් කරලා දැනට නවයෙන් කාල නොවැම්බර් දන්නවා රුපියල් විදිහ තුනක් හාරසිය හැට දෙකකට දෙක
කොල් තියෙනවා බිල්පත කරන්නේ හම්බ විසි දෙකයි තමයි ඩේට් තිබිලා තියෙනවා නොවැම්බර් බිලටත් පේමන්ට් ටික කළත් කනෙක්ෂන් ටික ඔටෝමැටිකලි ටික වෙලාවකින් වෙනවා එකක් හරි මැඩම්
පේමන්ට් එකක් කරලා තියෙන්නේ විදියේ විදිහට අට දෙපාරක් පස්සේ පේමන්ට් එකක් තාම භාවිතා කරලා නෑ පේ කරපු වන් එක කීයද කියලා දැනගන්න පුළුවන්ද මැඩම් මට ඩීටේල්ස් ටික දෙන්න 
පොළුවෙන් නමින්ද කොහොමද පේමන්ට් බැංක් මුල් එකෙන් මැඩම්ගේ ඩවුන් එක විදියක් මොනවාද ට්‍රාන්සක්ෂන් කරික්ෂණික රෙෆරන්ස් නම් කළත් මැඩම් ලබා ගත්තාද හෑන්ඩ් එක්ක අමවුන්ට් එකේනේ
එක රියම වුනොත් කරගෙන කළත් නමටද කරපු වෙලාව කියන්නම්"""

# Execute prediction calculation
prediction_report = engine.predict(noisy_input_sample)

# Display Clean Production Logs
print("=" * 50)
print(f"PREDICTED INTENT CATEGORY : {prediction_report['predicted_category']}")
print(f"SYSTEM CONFIDENCE SCORE   : {prediction_report['confidence_score'] * 100:.2f}%")
print(f"KEYWORD EVIDENCE FOUND    : {prediction_report['matched_evidence']}")
print("=" * 50)
print("RAW ENGINE SCORING MATRIX:")
for category, score in prediction_report['raw_scoring_matrix'].items():
    print(f" - {category.ljust(25)}: {score}")
print("=" * 50)

PREDICTED INTENT CATEGORY : Bill_Inquiries
SYSTEM CONFIDENCE SCORE   : 100.00%
KEYWORD EVIDENCE FOUND    : ["[Exact Keyword] 'රුපියල්'", "[Exact Keyword] 'බිල'", "[Fuzzy] 'එක'->'එකවුන්ට්' (0.80)", "[Fuzzy] 'මද'->'මුදලක්' (0.80)"]
RAW ENGINE SCORING MATRIX:
 - Bill_Inquiries           : 28.75
 - Telephone_Number_Request : 17.07
 - Fault_Inquiries          : 6.99
 - Technical_Assistant      : 8.67
 - Product_And_New_Service  : 4.79
 - Add_More_Data            : 8.76


In [4]:
# Initialize the fully offline engine instance
engine = HeuristicNoisyIntentClassifier()

# Your long transcript containing high conversational ambient noise
noisy_input_sample = """ආයුබෝවන් මම මාරු ක ට පුළුවනි සහයවන්න බිලටත් බලලා කියන්න මැඩම් එන්ටර් නම් එක අදාල කනෙක්ෂන් විස්තර නම්බර් එකයි විසින් හතයි විදිහ නම හයයි තියේ හෝම්
එකෙන් එකද මැඩම් කනෙක්ෂන් එකට අදාළවද අයිතිකරුගේ නම තැනකින් පුළුවන් [මතිමෙරදී] ඉන්නවා වලට කියන්න කරා ඇමතුම රීකනෙට් හාරදහස් සිරිත් කනෙක්ෂන් එකත් වෙලා නිසා ක්ලියර්
දැනුවත් වෙන නොවැම්බර් නැති දන්නවා රුපියල් දුන් හාරසිය දෙක මහත්මයා තියෙන්නේ දෙසැම්බර් විසි දෙකයි තමයි මේක බිලින් නවයෙන් හෙට රීපීට් ටික කරා කනෙක්ෂන් ටික තෝරන්න ජාතික
ටික වෙලාවකින් වෙනවා මැඩම් මන්ත චෙක් කරලා තියෙන්නේ මිස් විසා අට විදිහට ප්‍රොසීඩින් එකක් තාම [අභ්ෙිලානපේකරපු] ම් කියනකම් දැනගන්න පුළුවන්ද මට ට ඩීටේල්ස් ටික දෙන්න නමින්ද හ
ෝම් ම් රවුටර් එකෙන් මැඩම්ගේ ඩවුන් එකද ලින්ක් රෙෆරන්ස් නම් කළත් මැඩම් ලක්ෂිත ගෙවන්න ම් කළත් දාන්නම් ට කරපු වෙලාව කියන්න හතයි හතළිස් යොමුකරපු බැංක් එක මොකක්ද නේෂන් 
[ත්පැ්එකනහතහතලිස්පහනත] වෙලාව මෝඩ් ලක්ෂාන් සර් නම් එක දෙන්න පුළුවන්ද මට හැලෝ දැනගන්න කරන්නේ නේ පෝස්ට් නැහැ එතකොට අපි කරන්නේ බිල්පත ෂක්ෂ දාලා දැන් ක්ලියර් කී 
කී බිලින් පෙන්ඩින් මාසෙට අනිත් කට්ටිය කරන නේ මන්ත එකේනේ ටික වෙලාවකින් [මකික්වයිමටසාංශික්ෂ] නම් එකෙන් පුළුවන්ද [්්අතසීය] [හත්තතුනයිදෙසියානු] පහකින් අයිතිකරුගේ ට කතා 
කරන්නේ මට කරන්නම් නම් එක මෝඩ් මේ කෝල් කරලා නම් එකට මේ වෙල්කම් තැමි එකක් සොරි මම යොමු කරපු කියන එකට අදාළ කරන නම් එක මැඩම්ට කෝල් කරලා මබිටල් වන් එකට 
එක සත එකක් කනෙක්ෂන් කළත් නඩත්තු කී හතරක් ටික වෙලාවකින් කළේ බලාගන්න හරි එස්එල්ටී මොබිටෙල් මට මම හරි වෙනත් ගැනීමට අවශ්‍යද ලබා දුන් සේවය ඇගයීම විදිහ යොමු තැන්න 
ස්තූතියි මොබිටෙල් ස්තූතියි සත"""

# Execute prediction calculation
prediction_report = engine.predict(noisy_input_sample)

# Display Clean Production Logs
print("=" * 50)
print(f"PREDICTED INTENT CATEGORY : {prediction_report['predicted_category']}")
print(f"SYSTEM CONFIDENCE SCORE   : {prediction_report['confidence_score'] * 100:.2f}%")
print(f"KEYWORD EVIDENCE FOUND    : {prediction_report['matched_evidence']}")
print("=" * 50)
print("RAW ENGINE SCORING MATRIX:")
for category, score in prediction_report['raw_scoring_matrix'].items():
    print(f" - {category.ljust(25)}: {score}")
print("=" * 50)

PREDICTED INTENT CATEGORY : Bill_Inquiries
SYSTEM CONFIDENCE SCORE   : 100.00%
KEYWORD EVIDENCE FOUND    : ["[Exact Keyword] 'රුපියල්'", "[Fuzzy] 'රන'->'රිෆන්ඩ්' (0.79)", "[Exact Keyword] 'බිල'", "[Fuzzy] 'එක'->'එකවුන්ට්' (0.80)"]
RAW ENGINE SCORING MATRIX:
 - Bill_Inquiries           : 27.33
 - Telephone_Number_Request : 15.33
 - Fault_Inquiries          : 15.73
 - Technical_Assistant      : 6.26
 - Product_And_New_Service  : 11.15
 - Add_More_Data            : 13.52


In [5]:
# Initialize the fully offline engine instance
engine = HeuristicNoisyIntentClassifier()

# Your long transcript containing high conversational ambient noise
noisy_input_sample = """වන් මම හර්ෂි පුළුවනි බිල් [සහයවන්නලෝසගේියන්න්ලෑස්ටු] විදියක් එකද ට දින අලුතෙන් දාන්නම් ප්ලස් කරගෙන පොඩ්ඩක් සර් වෙප්සමිට් හෝම් ප්ලස් පකේජ් 
[රේැඳනසාරඳසනට] ස්තූතියි ලිමිටඩ් හෝම් ප්ලස් රෙන්ටල් බිල් නස් සියයි සර් ස්පීඩ් දුන් [රෙන්බිපිය්ටිනවා] ස්පීඩ් එක රෙන්ටර්ස් යනවා අප්ලෝඩ් එක රෙස්ට්‍රික්ශන් නැහැ අන්ලිමිටඩ් ඩියු කරන්නේ
පුළුවන් වොයිස් කෝල්ස් ලිමිටඩ් ගන්න ඉන්ස්ටන්ට්ලි හෝම් කියන ටයිම් බස්සන වෙනස් ස්පීඩ් එක මගේ දැන් තැන්ක්යු දෙන් එයාගෙන් අලුත් පහකින් [ැද්ෆ්ලෑස්න්ටිෆර්කත්තයන] ගාන දෙන් මේ 
අන්ලිමිටඩ් එහම ප්ලෑන්ස් මාරු ආයුබෝවන් ෆ්ලෑෂ් වෙන කරගෙන මාර්ගයෙන් පුළුවන් පුළුවන් කොහොමද අන්ලිමිටඩ් එහම සමාවෙන්න මාරු කරලා දන්න පුළුවන් කනෙක්ෂන් එක අයිතිකරුග
ේ අඩු කියන්න පුළුවන්ද [සටයයිහැටහ්තයහේටඋනිහනේ] සුභ කරන ඇමතුමේ දෙන්න ලයින් රිසිට් ස්තූතියි සහ දවස් අප්ඩේට් එවනවා නම් කෝල් වගේ මේ තියේ පකේජ් හතළිස් කරන්නේ පුළුවන් 
කොහොමද පකේජ් දින දෙකකට දේවල් ඇඩ් මේ මාසයේ බිල් එකට හේතුවෙන් වෙනස සමඟ අද දවස මේ පකේජ් තෝරන්න ඇතුලත් අලුත් පකේජ් කී අප්ඩේට් වෙන ගැන මට ඔය පස්සේ වුනොත්
මදි අඩු දැන් කවර් සමාවෙන්න පුළුවන් පුළුවන් නැහැ ගැනීමට කාලයකින් යනවා කීයක් දාලා ට මාර්ග අනිවාර්යයෙන් එකක් මොබයිල් ඩවුන්ග්‍රේඩ් එකක් රික්වෙස්ට් කෙරෙනවා ඉක්මනින්ම ලිමිටඩ් 
ෆයිබර් ප්ලෑන්ස් මුකුත් ට්‍රස්ට් එකක් නැහැ කළත් ඩවුන්ග්‍රේඩ් එකක් කියන්නම් බිලින් තමයි ඩවුන්ග්‍රේඩ් කරගන්න ගෙවන්න නේ සමාවෙන්න [විතරයසලෙසතමබි්තුතේ] [සැප] දවසක්"""

# Execute prediction calculation
prediction_report = engine.predict(noisy_input_sample)

# Display Clean Production Logs
print("=" * 50)
print(f"PREDICTED INTENT CATEGORY : {prediction_report['predicted_category']}")
print(f"SYSTEM CONFIDENCE SCORE   : {prediction_report['confidence_score'] * 100:.2f}%")
print(f"KEYWORD EVIDENCE FOUND    : {prediction_report['matched_evidence']}")
print("=" * 50)
print("RAW ENGINE SCORING MATRIX:")
for category, score in prediction_report['raw_scoring_matrix'].items():
    print(f" - {category.ljust(25)}: {score}")
print("=" * 50)

PREDICTED INTENT CATEGORY : Product_And_New_Service
SYSTEM CONFIDENCE SCORE   : 99.97%
KEYWORD EVIDENCE FOUND    : ["[Fuzzy] 'පක'->'පැකේජ්' (0.80)", "[Fuzzy] 'ලය'->'ලියුමක්' (0.79)", "[Ngram Sub-Word] 'ලිපියක්' (0.80)", "[Exact Keyword] 'වෙනස්'"]
RAW ENGINE SCORING MATRIX:
 - Bill_Inquiries           : 18.13
 - Telephone_Number_Request : 11.3
 - Fault_Inquiries          : 10.93
 - Technical_Assistant      : 5.9
 - Product_And_New_Service  : 26.36
 - Add_More_Data            : 16.41


In [6]:
# Initialize the fully offline engine instance
engine = HeuristicNoisyIntentClassifier()

# Your long transcript containing high conversational ambient noise
noisy_input_sample = """ආයුබෝවන් [මරානිමඩපොලනිපුට] සහය වන්න මිස් අපේ පියෝ ටීවී එක කරලයි ලයින් කරලයි දෙක වැඩ නැහැ දවස් ලයින් බිල්ලක් හේතුව දැනුවත් ඒක විදිහටම ආයේ
මේ නැවත කට්ටිය වෙලා තියෙනවා නයි පියෝ ටීවී එක කරික්ෂණික [එකිටකරුගෙනමෙන්ඩබ්ලි] රීපීට් ඒක කාලයකින් එක්ස්ට්‍රා එක වැඩ කරනවද චැනල්ස් වගේ වෙන ලයින් රෙඩ් හින්දා නවයෙන් ව
ෙලා තියෙනවා තියෙන්නේ එක ඇතුළත් කරන්නේ මට සම්බන්ධ තමයි නම් ම් හින්දා හතයි [හයයිරිඅසුතුනයි] අනේ එක්ක හිටියේ නැති සුබ තැන්න නස් මන්ත ස්තූතියි මේ දාන්නම් එක කව එකෙන් 
එක දන්න වෙලාවකින් නම්බර් එකක් දේවල් ඇමතීම තියේ සුබ දවසක් මාව මැඩම් සඳහා රැඳී සිටින්න"""

# Execute prediction calculation
prediction_report = engine.predict(noisy_input_sample)

# Display Clean Production Logs
print("=" * 50)
print(f"PREDICTED INTENT CATEGORY : {prediction_report['predicted_category']}")
print(f"SYSTEM CONFIDENCE SCORE   : {prediction_report['confidence_score'] * 100:.2f}%")
print(f"KEYWORD EVIDENCE FOUND    : {prediction_report['matched_evidence']}")
print("=" * 50)
print("RAW ENGINE SCORING MATRIX:")
for category, score in prediction_report['raw_scoring_matrix'].items():
    print(f" - {category.ljust(25)}: {score}")
print("=" * 50)

PREDICTED INTENT CATEGORY : Fault_Inquiries
SYSTEM CONFIDENCE SCORE   : 60.62%
KEYWORD EVIDENCE FOUND    : ["[Fuzzy] 'නව'->'නිවි නිවි' (0.77)", "[Ngram Sub-Word] 'වැඩකරන්නේනැහැ' (0.91)", "[Exact Keyword] 'චැනල්'", "[Fuzzy] 'ර'->'රතු' (0.80)"]
RAW ENGINE SCORING MATRIX:
 - Bill_Inquiries           : 12.04
 - Telephone_Number_Request : 12.93
 - Fault_Inquiries          : 14.04
 - Technical_Assistant      : 5.98
 - Product_And_New_Service  : 11.06
 - Add_More_Data            : 12.01
